# AMD 与 NVIDIA 原始 SEC 8-K 前瞻收入指引来源预检

冻结两家公司跨 2011/2026 各一份公告原件；只做来源、口径与 NAS 校验。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex,re
from pathlib import Path
from datetime import datetime,timezone
from bs4 import BeautifulSoup
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_amd_nvda_sec_guidance_original_probe_20261023';raw=root/'data/snapshots/stage1_amd_nvda_sec_guidance_original_probe_20261023';raw.mkdir(parents=True,exist_ok=True);plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and len(plan['exact_requests'])==4
state={'status':'started','GET_attempts':0,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');receipts=[];checks=[];failure=None
for i,spec in enumerate(plan['exact_requests'],1):
 body=raw/f'anchor_{i}.html';headers=raw/f'anchor_{i}.headers';state['GET_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','45','--max-filesize','2000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['request']['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=60);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
  receipt={'issuer':spec['issuer'],'anchor':spec['anchor'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:300]}
 except Exception as exc:receipt={'issuer':spec['issuer'],'anchor':spec['anchor'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'exception':str(exc)[:300]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None)
 receipts.append(receipt);(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists():failure=f'anchor_{i}_transport';break
 try:
  text=re.sub(r'\s+',' ',BeautifulSoup(body.read_bytes().decode('utf8','replace'),'html.parser').get_text(' ',strip=True));low=text.lower();assert ('nvidia' in low if spec['issuer']=='NVDA' else 'amd' in low or 'advanced micro devices' in low)
  outlook=re.search(r'\b(?:current\s+)?outlook\b',text,re.I);assert outlook
  section=text[outlook.start():outlook.start()+1300];assert 'revenue' in section.lower() and ('expect' in section.lower() or 'forecast' in section.lower())
  revenue=re.search(r'(?:revenue.{0,180}|.{0,50}revenue.{0,140})',section,re.I);assert revenue
  checks.append({'issuer':spec['issuer'],'anchor':spec['anchor'],'raw_sha256':sha(body),'text_chars':len(text),'outlook_revenue_excerpt':revenue.group(0)[:220],'outlook_section_excerpt':section[:450]});(card/'source_checks.json').write_text(json.dumps(checks,indent=2)+'\n')
 except Exception as exc:failure=f'anchor_{i}_semantic_{type(exc).__name__}_{str(exc)[:120]}';break
 time.sleep(plan['request']['seconds_between_requests'])
files=[{'name':x.name,'sha256':sha(x),'bytes':x.stat().st_size} for x in sorted(raw.iterdir()) if x.is_file()];manifest={'plan_sha256':sha(card/'plan.json'),'files':files,'receipts_sha256':sha(card/'capture_receipts.json')};(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='amd-nvda-guidance-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(raw/x['name'],arcname='raw/'+x['name'])
  for name in ('plan.json','capture_receipts.json','source_manifest.json'):t.add(card/name,arcname=name)
 digest=sha(archive);a=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert a['sha256']==digest and a['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='amd-nvda-guidance-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in files:
    m=t.extractfile('raw/'+x['name']);assert m and hashlib.sha256(m.read()).hexdigest()==x['sha256']
result={'status':'original_SEC_AMD_NVDA_early_and_recent_guidance_identity_passed_full_history_pending' if not failure and len(checks)==4 else 'bounded_original_SEC_AMD_NVDA_source_partial_or_failed','failure':failure,'plan_sha256':sha(card/'plan.json'),'GET_attempts':state['GET_attempts'],'checks':checks,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'first_public_clock_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,indent=2)+'\n');state.update(status='completed' if not failure else 'partial_or_failed',NAS_archive_sha256=digest);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'checks':len(checks),'NAS':digest})


{'status': 'original_SEC_AMD_NVDA_early_and_recent_guidance_identity_passed_full_history_pending', 'failure': None, 'GETs': 4, 'checks': 4, 'NAS': '5b30803141ac0368d71b6fa9b045360f9f04ac3c1ab7ef6d7305bd04cfe3781a'}


## 中文结论

四份 2011/2026 AMD/NVIDIA SEC 原始公告均可读且 NAS 恢复通过，原件格式横跨百分比与绝对金额。 完整决定见 `conclusion.md`。